# Music Tokenization with MidiTok

**CS 89.02 / MUS 14.05 — Music and AI, Week 4**

To apply language-model techniques (Transformers, GPT-style generation) to music,
we need to convert musical events into **discrete tokens** — analogous to words in text.

This notebook explores **MidiTok**, a library that provides multiple tokenization
strategies for MIDI data:

- **REMI** — Revamped MIDI-derived, includes explicit position tokens
- **TSD** — Time Shift Duration, compact time representation
- **Structured** — Fixed token-type ordering per note

The choice of tokenization affects sequence length, vocabulary size, and ultimately
how well a model can learn musical structure.

In [ ]:
!pip install miditok

## MIDI Basics

MIDI (Musical Instrument Digital Interface) represents music as a sequence of
**events** rather than audio samples:

- **Note On** (pitch, velocity, time)
- **Note Off** (pitch, time)
- **Control Change** (controller number, value)
- **Program Change** (instrument selection)
- **Tempo Change**, **Time Signature**, etc.

This event-based format is natural for symbolic music modeling but requires
further discretization (tokenization) for use with sequence models.

In [ ]:
import matplotlib.pyplot as plt
from miditok import REMI, TSD, Structured, TokenizerConfig
from symusic import Score, Note, Track, Tempo, TimeSignature


def create_sample_midi():
    """Create a simple MIDI file: C major scale followed by a chord."""
    score = Score(480)  # ticks per quarter note (symusic 0.6+ uses positional arg)

    # Add tempo and time signature
    score.tempos.append(Tempo(time=0, qpm=120))
    score.time_signatures.append(TimeSignature(time=0, numerator=4, denominator=4))

    track = Track(program=0, is_drum=False, name="Piano")

    # C major scale (quarter notes)
    scale_pitches = [60, 62, 64, 65, 67, 69, 71, 72]  # C4 to C5
    tpq = 480
    for i, pitch in enumerate(scale_pitches):
        track.notes.append(Note(
            time=i * tpq,
            duration=tpq,
            pitch=pitch,
            velocity=80
        ))

    # C major chord (whole note)
    chord_start = 8 * tpq
    for pitch in [60, 64, 67]:  # C, E, G
        track.notes.append(Note(
            time=chord_start,
            duration=4 * tpq,
            pitch=pitch,
            velocity=100
        ))

    score.tracks.append(track)
    return score


score = create_sample_midi()
print(f"Tracks: {len(score.tracks)}")
print(f"Notes: {len(score.tracks[0].notes)}")
print(f"Ticks per quarter: {score.tpq}")
print()
print("Notes:")
for note in score.tracks[0].notes:
    print(f"  pitch={note.pitch:3d}  vel={note.velocity:3d}  "
          f"time={note.time:5d}  dur={note.duration:4d}")

# Save for later use
score.dump_midi("sample.mid")

## Tokenization Strategies

Different tokenization strategies represent the same MIDI data in different ways:

### REMI (REvamped MIdi-derived)
Uses explicit **Bar** and **Position** tokens to encode timing.
Each note is represented by: `Bar → Position → Pitch → Velocity → Duration`.
This captures the metric structure explicitly.

### TSD (Time Shift Duration)
Encodes timing as **TimeShift** tokens (delta time since last event).
Each note is: `TimeShift → Pitch → Velocity → Duration`.
More compact than REMI for sparse music.

### Structured
Each note is encoded as a fixed-length tuple of tokens:
`(Pitch, Velocity, Duration, TimeShift)`.
This regular structure simplifies the model's task.

In [ ]:
# Configure tokenizers
config = TokenizerConfig(
    num_velocities=32,
    use_chords=False,
    use_programs=False,
)

# Create tokenizers
tokenizer_remi = REMI(config)
tokenizer_tsd = TSD(config)
tokenizer_struct = Structured(config)

# Tokenize the same MIDI with each strategy
tokens_remi = tokenizer_remi(score)
tokens_tsd = tokenizer_tsd(score)
tokens_struct = tokenizer_struct(score)

print("=" * 60)
print("REMI Tokenization")
print("=" * 60)
for tok in tokens_remi[0].tokens[:20]:
    print(f"  {tok}")
print(f"  ... ({len(tokens_remi[0].tokens)} tokens total)")

print()
print("=" * 60)
print("TSD Tokenization")
print("=" * 60)
for tok in tokens_tsd[0].tokens[:20]:
    print(f"  {tok}")
print(f"  ... ({len(tokens_tsd[0].tokens)} tokens total)")

print()
print("=" * 60)
print("Structured Tokenization")
print("=" * 60)
for tok in tokens_struct[0].tokens[:20]:
    print(f"  {tok}")
print(f"  ... ({len(tokens_struct[0].tokens)} tokens total)")

## Comparing Tokenizations

Key metrics for comparing tokenization strategies:

- **Sequence length** — shorter sequences are easier for Transformers (quadratic attention cost)
- **Vocabulary size** — smaller vocabularies mean fewer parameters in the embedding layer
- **Regularity** — fixed token-type patterns are easier for models to learn

In [ ]:
strategies = {
    'REMI': (tokenizer_remi, tokens_remi),
    'TSD': (tokenizer_tsd, tokens_tsd),
    'Structured': (tokenizer_struct, tokens_struct),
}

print(f"{'Strategy':<15} {'Seq Length':>12} {'Vocab Size':>12}")
print("-" * 42)
seq_lens = []
vocab_sizes = []
names = []
for name, (tokenizer, tokens) in strategies.items():
    seq_len = len(tokens[0].tokens)
    vocab_size = len(tokenizer.vocab)
    print(f"{name:<15} {seq_len:>12} {vocab_size:>12}")
    seq_lens.append(seq_len)
    vocab_sizes.append(vocab_size)
    names.append(name)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(names, seq_lens, color=['#4C72B0', '#DD8452', '#55A868'])
axes[0].set_title('Sequence Length')
axes[0].set_ylabel('Number of Tokens')

axes[1].bar(names, vocab_sizes, color=['#4C72B0', '#DD8452', '#55A868'])
axes[1].set_title('Vocabulary Size')
axes[1].set_ylabel('Number of Unique Tokens')

plt.tight_layout()
plt.show()

## Vocabulary and Token Statistics

Let us examine the distribution of token types within each tokenization
to understand what information each strategy emphasizes.

In [ ]:
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, (tokenizer, tokens)) in zip(axes, strategies.items()):
    # Extract token type prefixes (e.g., 'Pitch', 'Velocity', 'Position')
    token_types = []
    for tok in tokens[0].tokens:
        # Token strings are like 'Pitch_60', 'Velocity_80', etc.
        tok_str = str(tok)
        prefix = tok_str.split('_')[0]
        token_types.append(prefix)

    counts = Counter(token_types)
    labels = list(counts.keys())
    values = list(counts.values())

    ax.barh(labels, values)
    ax.set_title(f'{name} Token Types')
    ax.set_xlabel('Count')

plt.tight_layout()
plt.show()

# Summary table
print("\nToken type distributions:")
for name, (tokenizer, tokens) in strategies.items():
    token_types = [str(tok).split('_')[0] for tok in tokens[0].tokens]
    counts = Counter(token_types)
    print(f"\n{name}:")
    for ttype, count in counts.most_common():
        print(f"  {ttype:<15} {count:>4}")